# **SVD para Sistema de Recomendação**


Vamos realizar **Decomposição em Valores Singulares (SVD)** para construir um sistema de recomendação simples. O objetivo é prever notas que usuários dariam a filmes que ainda não avaliaram, usando apenas uma matriz pequena (6x6) de avaliações conhecidas.

**Dataset:** Matriz 6x6 com 6 usuários (linhas)e 6 filmes (colunas), onde cada célula contém uma nota de 0 a 5. O valor 0 representa ausência de avaliação (não "nota zero"

## **Objetivos**

* Demonstrar a matemática da SVD aplicada a recomendações

* Prever ratings faltantes a partir de padrões latentes

* Comparar implementações: NumPy vs SciPy

##  **Importação de Bibliotecas**

In [2]:
import pandas as pd
import numpy as np





##  **Construção da Matriz de Ratings**

In [3]:
# Matriz de notas (0 = não avaliado / sem nota)
#         Filme1  Filme2  Filme3  Filme4  Filme5  Filme6
dados = [
    [5,     3,      0,      1,      0,      4],   # User1
    [4,     0,      5,      0,      2,      0],   # User2
    [0,     5,      4,      0,      0,      3],   # User3
    [2,     0,      0,      5,      4,      0],   # User4
    [0,     4,      0,      3,      5,      2],   # User5
    [3,     0,      2,      0,      1,      5],   # User6
]


In [4]:
usuarios = ['User1', 'User2', 'User3', 'User4', 'User5', 'User6']
filmes   = ['Filme1', 'Filme2', 'Filme3', 'Filme4', 'Filme5', 'Filme6']

In [5]:
df = pd.DataFrame(dados, index=usuarios, columns=filmes)

print(df)

       Filme1  Filme2  Filme3  Filme4  Filme5  Filme6
User1       5       3       0       1       0       4
User2       4       0       5       0       2       0
User3       0       5       4       0       0       3
User4       2       0       0       5       4       0
User5       0       4       0       3       5       2
User6       3       0       2       0       1       5


## **Preparação da Matriz Numérica**

O **np.linalg.svd** exige um array NumPy, não uma lista Python.

In [6]:
A =dados.copy()

In [7]:
A = np.array(A)

# **Decomposição SVD**

In [8]:
U, S, VT =np.linalg.svd(A,full_matrices=False)

## **Matriz U (vetores singulares à esquerda)**

In [9]:
U

array([[-0.4846496 , -0.21990382,  0.03458461,  0.61560813, -0.56879452,
         0.11428212],
       [-0.35750261, -0.19553851,  0.59948505, -0.59331257, -0.16325968,
         0.30967841],
       [-0.41542793, -0.34470267, -0.56017129, -0.41744287, -0.09118353,
        -0.46068953],
       [-0.31381622,  0.65528057,  0.32676941,  0.01087221, -0.07480391,
        -0.59969549],
       [-0.44734719,  0.52600322, -0.41763565, -0.08271414,  0.17499771,
         0.55795675],
       [-0.40758575, -0.29751479,  0.2107888 ,  0.29629114,  0.77800052,
        -0.08862073]])

## **Valores singulares (S)**
Ordenados do maior para o menor — indicam a importância de cada fator latente:

In [10]:
S

array([12.21776392,  7.66462425,  6.13668901,  5.23791203,  2.84466786,
        1.67121819])

In [11]:
# Converte o vetor S em matriz diagonal — necessária para a reconstrução.
np.diag(S)

array([[12.21776392,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  7.66462425,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  6.13668901,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  5.23791203,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  2.84466786,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.67121819]])

## **Matriz VT (vetores singulares à direita)**

In [12]:
VT

array([[-0.46683241, -0.43547062, -0.34903247, -0.2779373 , -0.37769528,
        -0.50070581],
       [-0.19096257, -0.03642865, -0.38508513,  0.60466222,  0.59527335,
        -0.30651089],
       [ 0.62847709, -0.71172667,  0.19201195,  0.06771155,  0.10244258,
        -0.2156705 ],
       [ 0.3084069 , -0.10905997, -0.77201603,  0.08053339, -0.24063326,
         0.48227831],
       [-0.46143087, -0.5140531 ,  0.13181452, -0.14687863,  0.36111565,
         0.5945402 ],
       [ 0.2063591 ,  0.16229222, -0.28219385, -0.72421727,  0.55153653,
        -0.15086614]])

## **Reconstrução Completa (sem truncamento)**

In [13]:
Arec = U @ np.diag(S) @ VT
np.set_printoptions(suppress=True)

In [14]:
Arec

array([[ 5.,  3.,  0.,  1., -0.,  4.],
       [ 4., -0.,  5., -0.,  2.,  0.],
       [-0.,  5.,  4., -0.,  0.,  3.],
       [ 2.,  0., -0.,  5.,  4.,  0.],
       [-0.,  4., -0.,  3.,  5.,  2.],
       [ 3.,  0.,  2., -0.,  1.,  5.]])

##  **SVD Truncada (Redução de Dimensionalidade)**

* **k pequeno** → mais compressão, menos ruído, mas perda de informação

* **k grande** → mais fidelidade, mas mais ruído e overfittin

In [15]:
k=3 #numero de fatores (Valores sigulares). Reduzir a dimensionalidade

## Truncamento das matrizes

In [16]:
Uk = U[:,:k]  # # 6×3  → primeiras k colunas de U

In [17]:
Sk = np.diag(S[:k]) # # 3×3  → k maiores valores singulares

In [18]:
VTk = VT[:k,:]  ## 3×6  → primeiras k linhas de VT

In [19]:
# Reconstrução da matriz aproximada (elimina ruídos)
A_aprox = Uk @ Sk @ VTk

In [20]:
A_aprox

array([[ 3.21951929,  2.48891364,  2.75654298,  0.64098427,  1.25488057,
         3.43569174],
       [ 4.63734535, -0.66165681,  2.80805499,  0.55687337,  1.1344463 ,
         1.85298088],
       [ 0.71353005,  4.75314945,  2.12889086, -0.41959388, -0.00784796,
         4.09257767],
       [ 2.09106565,  0.05947703, -0.21080724,  4.23833269,  4.64330818,
        -0.05215245],
       [ 0.17089902,  4.05731858, -0.13695751,  3.78331778,  4.2016888 ,
         2.05365627],
       [ 3.57314872,  1.33096971,  2.864608  ,  0.09282146,  0.65593091,
         2.91337722]])

In [21]:
recomendacao = np.where(A==0, A_aprox, A)

In [22]:
recomendacao

array([[ 5.        ,  3.        ,  2.75654298,  1.        ,  1.25488057,
         4.        ],
       [ 4.        , -0.66165681,  5.        ,  0.55687337,  2.        ,
         1.85298088],
       [ 0.71353005,  5.        ,  4.        , -0.41959388, -0.00784796,
         3.        ],
       [ 2.        ,  0.05947703, -0.21080724,  5.        ,  4.        ,
        -0.05215245],
       [ 0.17089902,  4.        , -0.13695751,  3.        ,  5.        ,
         2.        ],
       [ 3.        ,  1.33096971,  2.        ,  0.09282146,  1.        ,
         5.        ]])

## **Comparativo**

In [23]:
print("Matriz original (com os filmes não avaliados):")
print(A)


Matriz original (com os filmes não avaliados):
[[5 3 0 1 0 4]
 [4 0 5 0 2 0]
 [0 5 4 0 0 3]
 [2 0 0 5 4 0]
 [0 4 0 3 5 2]
 [3 0 2 0 1 5]]


In [24]:

print("\nMatriz reconstruída (valores pelo modelo matemático ):") #
print(np.round(A_aprox, 2))


Matriz reconstruída (valores pelo modelo matemático ):
[[ 3.22  2.49  2.76  0.64  1.25  3.44]
 [ 4.64 -0.66  2.81  0.56  1.13  1.85]
 [ 0.71  4.75  2.13 -0.42 -0.01  4.09]
 [ 2.09  0.06 -0.21  4.24  4.64 -0.05]
 [ 0.17  4.06 -0.14  3.78  4.2   2.05]
 [ 3.57  1.33  2.86  0.09  0.66  2.91]]


In [25]:
print("\nMatriz com recomendação:") # notas para os valores sem avaliação (zeros)
print(np.round(recomendacao, 2))


Matriz com recomendação:
[[ 5.    3.    2.76  1.    1.25  4.  ]
 [ 4.   -0.66  5.    0.56  2.    1.85]
 [ 0.71  5.    4.   -0.42 -0.01  3.  ]
 [ 2.    0.06 -0.21  5.    4.   -0.05]
 [ 0.17  4.   -0.14  3.    5.    2.  ]
 [ 3.    1.33  2.    0.09  1.    5.  ]]


## **SVD com a biblioteca Scipy**

In [26]:

from scipy.linalg import svd

# Decomposição (SVD) using scipy
U, S, VT = svd(A, full_matrices=False)


In [27]:
print("Matriz U:")
print(U)

Matriz U:
[[-0.4846496  -0.21990382  0.03458461  0.61560813 -0.56879452  0.11428212]
 [-0.35750261 -0.19553851  0.59948505 -0.59331257 -0.16325968  0.30967841]
 [-0.41542793 -0.34470267 -0.56017129 -0.41744287 -0.09118353 -0.46068953]
 [-0.31381622  0.65528057  0.32676941  0.01087221 -0.07480391 -0.59969549]
 [-0.44734719  0.52600322 -0.41763565 -0.08271414  0.17499771  0.55795675]
 [-0.40758575 -0.29751479  0.2107888   0.29629114  0.77800052 -0.08862073]]


In [28]:
print("Matriz S:")
print(S)

Matriz S:
[12.21776392  7.66462425  6.13668901  5.23791203  2.84466786  1.67121819]


In [29]:
print("Matriz V:")
print(VT)

Matriz V:
[[-0.46683241 -0.43547062 -0.34903247 -0.2779373  -0.37769528 -0.50070581]
 [-0.19096257 -0.03642865 -0.38508513  0.60466222  0.59527335 -0.30651089]
 [ 0.62847709 -0.71172667  0.19201195  0.06771155  0.10244258 -0.2156705 ]
 [ 0.3084069  -0.10905997 -0.77201603  0.08053339 -0.24063326  0.48227831]
 [-0.46143087 -0.5140531   0.13181452 -0.14687863  0.36111565  0.5945402 ]
 [ 0.2063591   0.16229222 -0.28219385 -0.72421727  0.55153653 -0.15086614]]
